In [0]:
# Set catalog, schema, and volume variables using dbutils
dbutils.widgets.text("catalog", "rohitb_demo", "Catalog")
dbutils.widgets.text("schema", "sdp_airlines", "Schema")
dbutils.widgets.text("volume", "flight_data", "Volume")

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")

In [0]:

# Set catalog and schema
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"USE SCHEMA {schema}")

# Create volume if it doesn't exist
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

# Path to write CSV
csv_path = f"/Volumes/{catalog}/{schema}/{volume}/flights.csv"


In [0]:
%pip install faker

In [0]:
# Generate thousands of rows of realistic Spirit Airlines data using Faker
from faker import Faker
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, FloatType
from datetime import datetime, timedelta
import random

fake = Faker()
Faker.seed(42)
random.seed(42)

# Define Spirit Airlines routes (Departure, Arrival)
routes = [
    ("FLL", "ATL"), ("FLL", "DFW"), ("MCO", "LGA"), ("DFW", "LAS"),
    ("ORD", "FLL"), ("LAS", "MCO"), ("ATL", "LGA"), ("LGA", "DFW"),
    ("FLL", "LGA"), ("MCO", "ATL"), ("DFW", "ORD"), ("LAS", "ORD")
]

# Generate flights for each day in a month
start_date = datetime(2025, 11, 1)
num_days = 30
rows_per_day = 50  # Number of flights per day

data = []
for day in range(num_days):
    flight_date = start_date + timedelta(days=day)
    for _ in range(rows_per_day):
        route = random.choice(routes)
        dep, arr = route
        # Generate a random departure time on the given day
        hour = random.randint(5, 22)
        minute = random.choice([0, 15, 30, 45])
        dep_time = flight_date.replace(hour=hour, minute=minute, second=0)
        # Generate a flight number
        flight_number = f"NK{random.randint(100, 999)}"
        # Generate a realistic fare
        fare = round(random.uniform(49, 199), 2)
        data.append((flight_number, dep, arr, dep_time, fare))

schema = StructType([
    StructField("FlightNumber", StringType(), True),
    StructField("Departure", StringType(), True),
    StructField("Arrival", StringType(), True),
    StructField("DepartureTime", TimestampType(), True),
    StructField("Fare", FloatType(), True),
])

df = spark.createDataFrame(data, schema=schema)

df.display()

In [0]:
# Write to CSV
(
    df.coalesce(1)
      .write
      .mode("overwrite")
      .option("header", True)
      .csv(csv_path)
)

print(f"Generated {len(data)} rows of Spirit Airlines flight data written to: {csv_path}")

display(df.limit(20))